In [1]:
!pip install pydub

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Pre - Processed Folder Making

In [3]:
import os

base_path = "/content/drive/MyDrive/IAPC Project"

preprocessed_path = os.path.join(base_path, "Pre - Processed Dataset")

os.makedirs(preprocessed_path, exist_ok=True)

subfolders = [
    ("AI Voices", "AI English"),
    ("AI Voices", "AI Hindi"),
    ("Human Voices", "Human English"),
    ("Human Voices", "Human Hindi")
]

for main, sub in subfolders:
    path = os.path.join(preprocessed_path, main, sub)
    os.makedirs(path, exist_ok=True)

# Pre - Processing

## .wav format conversion

In [4]:
import os
from pydub import AudioSegment

input_base = "/content/drive/MyDrive/IAPC Project/Dataset"

output_base = "/content/drive/MyDrive/IAPC Project/Pre - Processed Dataset"

folders = [
    ("AI Voices", "AI English"),
    ("AI Voices", "AI Hindi"),
    ("Human Voices", "Human English"),
    ("Human Voices", "Human Hindi")
]

for main_folder, sub_folder in folders:

    input_folder = os.path.join(input_base, main_folder, sub_folder)
    output_folder = os.path.join(output_base, main_folder, sub_folder)

    files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.mp3', '.wav', '.flac', '.m4a'))]

    for file in files:
        input_path = os.path.join(input_folder, file)

        output_filename = os.path.splitext(file)[0] + ".wav"
        output_path = os.path.join(output_folder, output_filename)

        try:
            audio = AudioSegment.from_file(input_path)
            audio.export(output_path, format="wav")

        except Exception as e:
            print(f"Error: {input_path}")

    print(f"Converted: {main_folder} → {sub_folder}")

print("All files converted to WAV and saved!")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Converted: AI Voices → AI English
Converted: AI Voices → AI Hindi
Converted: Human Voices → Human English
Converted: Human Voices → Human Hindi
All files converted to WAV and saved!


## Resampling to 16kHz

In [5]:
import os
import librosa
import soundfile as sf

base_path = "/content/drive/MyDrive/IAPC Project/Pre - Processed Dataset"

TARGET_SR = 16000

folders = [
    ("AI Voices", "AI English"),
    ("AI Voices", "AI Hindi"),
    ("Human Voices", "Human English"),
    ("Human Voices", "Human Hindi")
]

for main_folder, sub_folder in folders:

    folder_path = os.path.join(base_path, main_folder, sub_folder)

    files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]

    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            y, sr = librosa.load(file_path, sr=None)

            if sr != TARGET_SR:
                y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)

            sf.write(file_path, y, TARGET_SR)

        except Exception as e:
            print(f"Error: {file_path}")

    print(f"Resampled: {main_folder} → {sub_folder}")

print("All files resampled to 16kHz!")

Resampled: AI Voices → AI English
Resampled: AI Voices → AI Hindi
Resampled: Human Voices → Human English
Resampled: Human Voices → Human Hindi
All files resampled to 16kHz!


## Mono (Single Audio Channel) Conversion

In [6]:
import os
import librosa
import soundfile as sf

base_path = "/content/drive/MyDrive/IAPC Project/Pre - Processed Dataset"

folders = [
    ("AI Voices", "AI English"),
    ("AI Voices", "AI Hindi"),
    ("Human Voices", "Human English"),
    ("Human Voices", "Human Hindi")
]

for main_folder, sub_folder in folders:

    folder_path = os.path.join(base_path, main_folder, sub_folder)

    files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]

    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            y, sr = librosa.load(file_path, sr=None, mono=True)

            sf.write(file_path, y, sr)

        except Exception as e:
            print(f"Error: {file_path}")

    print(f"Converted to mono: {main_folder} → {sub_folder}")

print("All files converted to MONO!")

Converted to mono: AI Voices → AI English
Converted to mono: AI Voices → AI Hindi
Converted to mono: Human Voices → Human English
Converted to mono: Human Voices → Human Hindi
All files converted to MONO!


## Amplitude Normalization

In [7]:
import os
import librosa
import soundfile as sf
import numpy as np

base_path = "/content/drive/MyDrive/IAPC Project/Pre - Processed Dataset"

folders = [
    ("AI Voices", "AI English"),
    ("AI Voices", "AI Hindi"),
    ("Human Voices", "Human English"),
    ("Human Voices", "Human Hindi")
]

for main_folder, sub_folder in folders:

    folder_path = os.path.join(base_path, main_folder, sub_folder)

    files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]

    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            y, sr = librosa.load(file_path, sr=None)

            max_val = np.max(np.abs(y))
            if max_val > 0:
                y = y / max_val

            sf.write(file_path, y, sr)

        except Exception as e:
            print(f"Error: {file_path}")

    print(f"Normalized: {main_folder} → {sub_folder}")

print("All files normalized successfully!")

Normalized: AI Voices → AI English
Normalized: AI Voices → AI Hindi
Normalized: Human Voices → Human English
Normalized: Human Voices → Human Hindi
All files normalized successfully!


## Padding + 30 secs Clipping

In [8]:
import os
import librosa
import soundfile as sf
import numpy as np

base_path = "/content/drive/MyDrive/IAPC Project/Pre - Processed Dataset"

TARGET_SR = 16000
TARGET_DURATION = 30
TARGET_LENGTH = TARGET_SR * TARGET_DURATION

folders = [
    ("AI Voices", "AI English"),
    ("AI Voices", "AI Hindi"),
    ("Human Voices", "Human English"),
    ("Human Voices", "Human Hindi")
]

for main_folder, sub_folder in folders:

    folder_path = os.path.join(base_path, main_folder, sub_folder)

    files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]

    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            y, sr = librosa.load(file_path, sr=TARGET_SR)

            if len(y) > TARGET_LENGTH:
                y = y[:TARGET_LENGTH]

            elif len(y) < TARGET_LENGTH:
                pad_length = TARGET_LENGTH - len(y)
                y = np.pad(y, (0, pad_length))

            sf.write(file_path, y, TARGET_SR)

        except Exception as e:
            print(f"Error: {file_path}")

    print(f"Processed (30s fixed): {main_folder} → {sub_folder}")

print("All audio standardized to 30 seconds!")

Processed (30s fixed): AI Voices → AI English
Processed (30s fixed): AI Voices → AI Hindi
Processed (30s fixed): Human Voices → Human English
Processed (30s fixed): Human Voices → Human Hindi
All audio standardized to 30 seconds!


## Pre - emphasis

In [10]:
import os
import librosa
import soundfile as sf
import numpy as np

base_path = "/content/drive/MyDrive/IAPC Project/Pre - Processed Dataset"

alpha = 0.97

folders = [
    ("AI Voices", "AI English"),
    ("AI Voices", "AI Hindi"),
    ("Human Voices", "Human English"),
    ("Human Voices", "Human Hindi")
]

for main_folder, sub_folder in folders:

    folder_path = os.path.join(base_path, main_folder, sub_folder)

    files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]

    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            y, sr = librosa.load(file_path, sr=None)

            y_preemph = np.append(y[0], y[1:] - alpha * y[:-1])

            sf.write(file_path, y_preemph, sr)

        except Exception as e:
            print(f"Error: {file_path}")

    print(f"Pre-emphasis applied: {main_folder} → {sub_folder}")

print("Pre-emphasis applied to all files!")

Pre-emphasis applied: AI Voices → AI English
Pre-emphasis applied: AI Voices → AI Hindi
Pre-emphasis applied: Human Voices → Human English
Pre-emphasis applied: Human Voices → Human Hindi
Pre-emphasis applied to all files!


## DC Offset removal

In [11]:
import os
import librosa
import soundfile as sf
import numpy as np

base_path = "/content/drive/MyDrive/IAPC Project/Pre - Processed Dataset"

folders = [
    ("AI Voices", "AI English"),
    ("AI Voices", "AI Hindi"),
    ("Human Voices", "Human English"),
    ("Human Voices", "Human Hindi")
]

for main_folder, sub_folder in folders:

    folder_path = os.path.join(base_path, main_folder, sub_folder)

    files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]

    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            y, sr = librosa.load(file_path, sr=None)

            y = y - np.mean(y)

            sf.write(file_path, y, sr)

        except Exception as e:
            print(f"Error: {file_path}")

    print(f"DC Offset removed: {main_folder} → {sub_folder}")

print("DC Offset removal completed!")

DC Offset removed: AI Voices → AI English
DC Offset removed: AI Voices → AI Hindi
DC Offset removed: Human Voices → Human English
DC Offset removed: Human Voices → Human Hindi
DC Offset removal completed!


# CSV File generation

In [9]:
import os
import pandas as pd
import librosa

base_path = "/content/drive/MyDrive/IAPC Project/Pre - Processed Dataset"

data = []

folders = [
    ("AI", "English", os.path.join(base_path, "AI Voices", "AI English")),
    ("AI", "Hindi", os.path.join(base_path, "AI Voices", "AI Hindi")),
    ("Human", "English", os.path.join(base_path, "Human Voices", "Human English")),
    ("Human", "Hindi", os.path.join(base_path, "Human Voices", "Human Hindi"))
]

for label, language, folder_path in folders:

    if not os.path.exists(folder_path):
        print(f"Folder not found: {folder_path}")
        continue

    files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]

    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            duration = librosa.get_duration(path=file_path)

            data.append({
                "file_path": file_path,
                "label": label,
                "language": language,
                "duration": round(duration, 2)
            })

        except:
            print(f"⚠️ Error: {file_path}")

df = pd.DataFrame(data)

csv_path = os.path.join(base_path, "preprocessed_audio_dataset.csv")

df.to_csv(csv_path, index=False)

✅ CSV created successfully!
📁 Saved at: /content/drive/MyDrive/IAPC Project/Pre - Processed Dataset/preprocessed_audio_dataset.csv
📊 Total files: 1768
